In [46]:
! pip install agent-framework-openai  -U

In [47]:
# 📦 Import Core Libraries for Agent Design Patterns
import os                     # Environment variable access for configuration management
from random import randint    # Random selection utilities for tool functionality

from dotenv import load_dotenv  # Secure environment configuration loading

In [48]:
# 🤖 Import Microsoft Agent Framework Components  
# ChatAgent: Core agent orchestration class following factory pattern
# OpenAIChatClient: GitHub Models integration following adapter pattern
from agent_framework import Agent, tool, AgentRunInputs, ResponseStream
from agent_framework.openai import OpenAIChatCompletionClient
from typing import Annotated  # Type annotations for enhanced code clarity and tool integration

In [49]:
# 🔧 Configuration Loading Pattern
# Implement configuration management pattern for secure credential handling
# This follows the external configuration principle for cloud-native applications
load_dotenv()

True

In [50]:
openai_chat_client = OpenAIChatCompletionClient(base_url=os.environ.get("GITHUB_ENDPOINT"), 
                                      api_key=os.environ.get("GITHUB_TOKEN"), 
                                      model=os.environ.get("GITHUB_MODEL_ID"))
                                      #handoffs=[difficulty_agent])

In [51]:
# RUN QUERY
# -----------------------------
import requests
import pandas as pd

@tool
def get_rutgers_courses(subject_code: str = "954"):
    """
    Fetches real-time course data from the Rutgers SOC API.
    subject_code: The 3-digit subject code (e.g., '954' for Data Science, '198' for CS).
    """
    url = "https://classes.rutgers.edu/soc/api/courses.json?year=2024&term=9&campus=NB"
    try:
        response = requests.get(url)
        all_courses = response.json()
        
        # Filter for the requested subject and parse into a readable format
        matching_classes = []
        for course in all_courses:
            if course['subject'] == subject_code:
                # Get the first section's meeting time for simplicity
                section = course['sections'][0]
                meeting = section['meetingTimes'][0] if section['meetingTimes'] else {}
                
                matching_classes.append({
                    "title": course['title'],
                    "course_id": course['courseString'],
                    "instructor": section['instructorsText'],
                    "time": f"{meeting.get('startTime', 'N/A')} - {meeting.get('endTime', 'N/A')}",
                    "day": meeting.get('meetingDay', 'N/A'),
                    "campus": meeting.get('campusName', 'N/A')
                })
        
        if not matching_classes:
            return f"No classes found for subject {subject_code}."
            
        return matching_classes[:10]  # Return top 10 to keep the AI's context clean
    except Exception as e:
        return f"Error fetching data: {str(e)}"


In [52]:
# AGENT INSTRUCTIONS
# -----------------------------
AGENT_INSTRUCTIONS = AGENT_INSTRUCTIONS = """
You are the Rutgers Schedule Assistant.
1. Use 'get_rutgers_courses' to find classes. Use '954' for Data Science/Stats or '198' for CS.
2. When the user gives a time preference (e.g., 'Morning'), filter the API results for times before 1200.
3. Present the schedule as a clear Markdown table.
"""
#4. If the user mentions being 'worried' or 'overwhelmed', transfer to the DifficultyExpert.

In [53]:
# CREATE AGENT
# -----------------------------
coordinator_agent = Agent(
    name="ScheduleCoordinator",
    client=openai_chat_client,
    instructions=AGENT_INSTRUCTIONS,
    tools=[get_rutgers_courses],
    #handoffs=[difficulty_expert]
)

In [54]:
thread = coordinator_agent.create_session()

In [55]:
# RUNNING THE INTERACTION
    
query = "I'm a CS major. I prefer morning classes, but I'm worried about the time of the classes."
    #query = "I'm a CS major. I prefer morning classes, but I'm worried about taking too many hard ones."
    
# Run the coordinator
response = await coordinator_agent.run(query, session=thread)
print(response.text)

Here are the morning CS classes (before 12:00) available to you:

| Course ID      | Title           | Instructor       | Time       | Day | Campus       |
|----------------|-----------------|------------------|------------|-----|--------------|
| 01:198:107     | COMPUT MATH & SCIENC | MOTTO, DOUGLAS   | 05:40-07:00 | T   | BUSCH        |
| 01:198:142     | DATA 101        | IMIELINSKI       | 10:20-11:40 | W   | BUSCH        |
| 01:198:170     | COMP APPS-BUSINESS | LAU, ARNOLD      | 05:55-06:50 | M   | COLLEGE AVE  |

Would you like to see more classes or information on any specific course?
